Facebook data downloaded from Humanitarian Data Exchange on 17th January 2025.
Links for 2020 and 2021-2022 data are: https://data.humdata.org/dataset/c3429f0e-651b-4788-bb2f-4adbf222c90e/resource/55a51014-0d27-49ae-bf92-c82a570c2c6c/download/movement-range-data-2022-05-22.zip
and https://data.humdata.org/dataset/c3429f0e-651b-4788-bb2f-4adbf222c90e/resource/3d77ce5c-ab6d-4864-b8a2-c8bafffac4f3/download/movement-range-data-2020-03-01-2020-12-31.zip
which are accessible from:
https://data.humdata.org/dataset/movement-range-maps?

In [ ]:
import pycountry
import polars as pl
import json
import pandas as pd

from emu_renewal.inputs import RAW_MOB_PATH, DATA_PATH

In [ ]:
data_col = "all_day_bing_tiles_visited_relative_change"
country = "Spain"
iso3 = pycountry.countries.lookup(country).alpha_3

# Should be 1 for most European countries, 2 for most others
gadm_level = 1

In [ ]:
# Compile Facebook data
data20 = pl.read_csv(RAW_MOB_PATH / "movement-range-data-2020-03-01--2020-12-31.txt", separator="\t", schema_overrides={"ds": pl.datatypes.Date})
data21_22 = pl.read_csv(RAW_MOB_PATH / "movement-range-2022-05-22.txt", separator="\t", schema_overrides={"ds": pl.datatypes.Date})
fb_data = pl.concat([data20, data21_22])
country_mobility = fb_data.filter(pl.col("country") == iso3)

# Get country population data produced by gridded population notebook
country_pop_data = json.load(open(DATA_PATH/f"population/gadm_est/{iso3}_{gadm_level}.json",'r'))

In [ ]:
# Calculate weighted average over patches
country_mob_series = pd.Series(0.0, index=country_mobility["ds"].unique(), dtype=float)
total_pop = 0.0
for pid in country_pop_data:
    if pid in country_mobility["polygon_id"]:
        cur_data = country_mobility.filter(pl.col("polygon_id") == pid)
        mob_series = pd.Series(index=cur_data["ds"].unique(), data=cur_data[data_col]).dropna()
        region_pop = country_pop_data[pid]
        country_mob_series += mob_series.reindex(country_mob_series.index, method="nearest") * region_pop
        total_pop += region_pop
weighted_country_mob = country_mob_series / total_pop

In [ ]:
weighted_country_mob.to_csv(DATA_PATH / f"mobility/{iso3}_fbmob_data.csv")